In [42]:
import time
import pandas as pd 
from nba_api.stats.endpoints import boxscoretraditionalv3, boxscoreadvancedv3

SLEEP_SECONDS = 3
MAX_RETRIES = 3
CHECKPOINT_EVERY = 50

In [43]:
def pull_starters(game_id):
    box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
    players = box.get_data_frames()[0]

    is_starter = (players["position"].notna()) & (players["position"] != "")
    starters = players[is_starter].copy()

    starters = starters[
        ["gameId", "teamId", "personId", "firstName", "familyName", "position"]
    ]
    starters = starters.rename(
        columns={
            "gameId": "GAME_ID",
            "teamId": "TEAM_ID",
            "personId": "PLAYER_ID",
            "firstName": "FIRST_NAME",
            "familyName": "LAST_NAME",
            "position": "START_POSITION"
        }
    )
    return starters

In [44]:
def pull_team_advanced(game_id): 
    box = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=game_id)
    team_stats = box.get_data_frames()[1] 
    
    team_stats = team_stats[
        [
            "gameId",
            "teamId",
            "teamTricode",
            "offensiveRating",
            "defensiveRating",
            "pace",
            "possessions"
        ]
    ]
    team_stats = team_stats.rename(
        columns={
            "gameId": "GAME_ID",
            "teamId": "TEAM_ID",
            "teamTricode": "TEAM_TRICODE",
            "offensiveRating": "OFF_RATING",
            "defensiveRating": "DEF_RATING",
            "pace": "PACE",
            "possessions": "POSSESSIONS"
        }
    )
    return team_stats

In [45]:
def main():
    game_index = pd.read_csv("game_index.csv", dtype={"GAME_ID": str})
    unique_game_ids = game_index["GAME_ID"].unique()
    print(f"Found {len(unique_game_ids)} unique games to process.")

    starters_frames = []
    advanced_frames = []
    failed_ids = []

    for i, game_id in enumerate(unique_game_ids):
        try:
            starters_frames.append(pull_starters(game_id))
            time.sleep(SLEEP_SECONDS)
            advanced_frames.append(pull_team_advanced(game_id))
            time.sleep(SLEEP_SECONDS)
        except Exception as e:
            print(f" FAILED on {game_id}: {e}")
            failed_ids.append(game_id)
            continue

        if (i + 1) % CHECKPOINT_EVERY == 0:
            print(f"Processed {i + 1}/{len(unique_game_ids)} games...")
            pd.concat(starters_frames, ignore_index=True).to_csv(
                "starters_checkpoint.csv", index=False
            )
            pd.concat(advanced_frames, ignore_index=True).to_csv(
                "advanced_checkpoint.csv", index=False
            )
    starters_result = pd.concat(starters_frames, ignore_index=True)
    advanced_result = pd.concat(advanced_frames, ignore_index=True)

    starters_result.to_csv("starters_full.csv", index=False)
    advanced_result.to_csv("advanced_full.csv", index=False)

    print(f"\nSaved {len(starters_result)} starter rows to starters_full.csv")
    print(f"Saved {len(advanced_result)} team-game advanced rows to advanced_full.csv")

    if failed_ids:
        pd.Series(failed_ids).to_csv("failed_game_ids.csv", index=False)
        print(f"\n{len(failed_ids)} games failed - see failed_games_ids.csv")
        print("Re-run targeting just those IDs to retry.")

    print("Done.")

In [41]:
main()

Found 11497 unique games to process.
Processed 50/11497 games...
Processed 100/11497 games...
Processed 150/11497 games...
Processed 200/11497 games...
Processed 250/11497 games...
Processed 300/11497 games...
Processed 350/11497 games...
Processed 400/11497 games...
Processed 450/11497 games...
Processed 500/11497 games...
Processed 550/11497 games...
Processed 600/11497 games...
 FAILED on 0021600596: Expecting value: line 1 column 1 (char 0)
Processed 650/11497 games...
Processed 700/11497 games...
Processed 750/11497 games...
Processed 800/11497 games...
Processed 850/11497 games...
Processed 900/11497 games...
Processed 950/11497 games...
Processed 1000/11497 games...
Processed 1050/11497 games...
Processed 1100/11497 games...
Processed 1150/11497 games...
Processed 1200/11497 games...
Processed 1250/11497 games...
Processed 1300/11497 games...
Processed 1350/11497 games...
Processed 1400/11497 games...
Processed 1450/11497 games...
Processed 1500/11497 games...
Processed 1550/114

KeyboardInterrupt: 

In [47]:
"""
Retry script: resumes from your existing checkpoint files instead of
starting over, slows down request pace, adds browser-like headers,
and retries each failed call a few times before giving up.

Run this in Jupyter the same way as before: imports/config cell,
then function cells, then main() in its own final cell.

Requires starters_checkpoint.csv and advanced_checkpoint.csv to already
exist in this folder (they should, from your earlier run). If they
don't exist, this script just starts fresh.
"""

import time
import pandas as pd
from nba_api.stats.endpoints import boxscoretraditionalv3, boxscoreadvancedv3

SLEEP_SECONDS = 3.0      # slower pace than before (was 0.6)
MAX_RETRIES = 3          # retry each failed call this many times
CHECKPOINT_EVERY = 25    # save more often since each game now takes longer

CUSTOM_HEADERS = {
    "Host": "stats.nba.com",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "x-nba-stats-origin": "stats",
    "x-nba-stats-token": "true",
    "Connection": "keep-alive",
    "Referer": "https://stats.nba.com/",
    "Origin": "https://www.nba.com",
}


def pull_starters(game_id):
    box = boxscoretraditionalv3.BoxScoreTraditionalV3(
        game_id=game_id, headers=CUSTOM_HEADERS, timeout=60
    )
    players = box.get_data_frames()[0]

    is_starter = (players["position"].notna()) & (players["position"] != "")
    starters = players[is_starter].copy()

    starters = starters[
        ["gameId", "teamId", "personId", "firstName", "familyName", "position"]
    ]
    starters = starters.rename(
        columns={
            "gameId": "GAME_ID",
            "teamId": "TEAM_ID",
            "personId": "PLAYER_ID",
            "firstName": "FIRST_NAME",
            "familyName": "LAST_NAME",
            "position": "START_POSITION",
        }
    )
    return starters


def pull_team_advanced(game_id):
    box = boxscoreadvancedv3.BoxScoreAdvancedV3(
        game_id=game_id, headers=CUSTOM_HEADERS, timeout=60
    )
    team_stats = box.get_data_frames()[1]

    team_stats = team_stats[
        [
            "gameId",
            "teamId",
            "teamTricode",
            "offensiveRating",
            "defensiveRating",
            "pace",
            "possessions",
        ]
    ]
    team_stats = team_stats.rename(
        columns={
            "gameId": "GAME_ID",
            "teamId": "TEAM_ID",
            "teamTricode": "TEAM_TRICODE",
            "offensiveRating": "OFF_RATING",
            "defensiveRating": "DEF_RATING",
            "pace": "PACE",
            "possessions": "POSSESSIONS",
        }
    )
    return team_stats


def pull_with_retry(pull_func, game_id):
    """Try a pull function up to MAX_RETRIES times, waiting longer each time."""
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return pull_func(game_id)
        except Exception as e:
            last_error = e
            wait_time = SLEEP_SECONDS * (attempt + 1)
            time.sleep(wait_time)
    raise last_error


def load_existing(path):
    try:
        return pd.read_csv(path, dtype={"GAME_ID": str})
    except FileNotFoundError:
        return pd.DataFrame()


def main():
    game_index = pd.read_csv("game_index.csv", dtype={"GAME_ID": str})
    all_game_ids = game_index["GAME_ID"].unique().tolist()

    existing_starters = load_existing("starters_checkpoint.csv")
    existing_advanced = load_existing("advanced_checkpoint.csv")

    done_starters = (
        set(existing_starters["GAME_ID"].unique()) if len(existing_starters) else set()
    )
    done_advanced = (
        set(existing_advanced["GAME_ID"].unique()) if len(existing_advanced) else set()
    )
    # Only count a game as done if BOTH pulls succeeded for it
    done_ids = done_starters & done_advanced

    remaining_ids = [g for g in all_game_ids if g not in done_ids]
    print(f"{len(done_ids)} games already completed previously.")
    print(f"{len(remaining_ids)} games remaining to pull.")

    starters_frames = [existing_starters] if len(existing_starters) else []
    advanced_frames = [existing_advanced] if len(existing_advanced) else []
    failed_ids = []

    for i, game_id in enumerate(remaining_ids):
        try:
            starters_result = pull_with_retry(pull_starters, game_id)
            advanced_result = pull_with_retry(pull_team_advanced, game_id)
            starters_frames.append(starters_result)
            advanced_frames.append(advanced_result)
        except Exception as e:
            print(f" FAILED on {game_id}: {e}")
            failed_ids.append(game_id)

        time.sleep(SLEEP_SECONDS)

        if (i + 1) % CHECKPOINT_EVERY == 0:
            print(f"Processed {i + 1}/{len(remaining_ids)} remaining games...")
            pd.concat(starters_frames, ignore_index=True).to_csv(
                "starters_checkpoint.csv", index=False
            )
            pd.concat(advanced_frames, ignore_index=True).to_csv(
                "advanced_checkpoint.csv", index=False
            )

    final_starters = pd.concat(starters_frames, ignore_index=True)
    final_advanced = pd.concat(advanced_frames, ignore_index=True)

    final_starters.to_csv("starters_full.csv", index=False)
    final_advanced.to_csv("advanced_full.csv", index=False)

    print(f"\nSaved {len(final_starters)} starter rows total to starters_full.csv")
    print(f"Saved {len(final_advanced)} team-game advanced rows total to advanced_full.csv")

    if failed_ids:
        pd.Series(failed_ids).to_csv("failed_game_ids.csv", index=False)
        print(f"\n{len(failed_ids)} games still failing after retries - see failed_game_ids.csv")

    print("Done.")

In [48]:
main()

2420 games already completed previously.
9077 games remaining to pull.
 FAILED on 0021600596: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700602: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700593: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700594: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700596: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700595: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700597: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700598: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
 FAILED on 0021700591: HTTPSConnectionPool(host='

KeyboardInterrupt: 

In [50]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")

print("Path to dataset files:", path)

100%|██████████| 1.03G/1.03G [00:20<00:00, 54.1MB/s]

Extracting files...


Path to dataset files: /Users/zekesnyder/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515


In [51]:
import os
import pandas as pd

path = "/Users/zekesnyder/.cache/kagglehub/datasets/eoinamoore/historical-nba-data-and-player-box-scores/versions/515"

# List every file in the dataset
for f in os.listdir(path):
    print(f)

LeagueSchedule25_26.csv
TeamStatisticsExtended.csv
PlayByPlay.parquet
PlayerStatisticsExtended.csv
TeamStatistics.csv
LeagueSchedule24_25.csv
PlayerStatistics.csv
Players.csv
TeamHistories.csv
Games.csv


In [52]:
for fname in ["Players.csv", "PlayerStatistics.csv", "TeamStatistics.csv", "TeamStatisticsExtended.csv"]:
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print('='*60)
    df = pd.read_csv(os.path.join(path, fname), nrows=5)
    print(f"Columns ({len(df.columns)} total):")
    print(df.columns.tolist())
    print("\nSample rows:")
    print(df.head(3))


FILE: Players.csv
Columns (20 total):
['personId', 'firstName', 'lastName', 'birthDate', 'school', 'country', 'heightInches', 'bodyWeightLbs', 'jersey', 'guard', 'forward', 'center', 'dleagueFlag', 'nbaFlag', 'gamesPlayedFlag', 'draftYear', 'draftRound', 'draftNumber', 'fromYear', 'toYear']

Sample rows:
    personId firstName    lastName   birthDate      school country  \
0  196294534   Alberto      Abalde         NaN         NaN     NaN   
1      76001      Alaa   Abdelnaby  1968-06-24        Duke     USA   
2      76002      Zaid  Abdul-Aziz  1946-04-07  Iowa State     USA   

   heightInches  bodyWeightLbs  jersey  guard  forward  center  dleagueFlag  \
0           NaN            NaN     NaN      0        0       0          NaN   
1          82.0          240.0    30.0      0        1       0          0.0   
2          81.0          235.0    54.0      0        0       1          0.0   

   nbaFlag  gamesPlayedFlag  draftYear  draftRound  draftNumber  fromYear  \
0      NaN        

In [54]:
df = pd.read_csv(
    os.path.join(path, "PlayerStatistics.csv"),
    usecols=["startingPosition"],
    dtype=str,
    low_memory=False,
)
print(df["startingPosition"].value_counts(dropna=False))


startingPosition
NaN    1097141
G       307399
F       196781
C        68601
Name: count, dtype: int64


In [55]:
print(pd.read_csv(os.path.join(path, "Players.csv"), nrows=1).T)

                         0
personId         196294534
firstName          Alberto
lastName            Abalde
birthDate              NaN
school                 NaN
country                NaN
heightInches           NaN
bodyWeightLbs          NaN
jersey                 NaN
guard                    0
forward                  0
center                   0
dleagueFlag            NaN
nbaFlag                NaN
gamesPlayedFlag        NaN
draftYear              NaN
draftRound             NaN
draftNumber            NaN
fromYear               NaN
toYear                 NaN
